# 02. Basic Parsing and Guardrails

**Purpose:** Core prompt-engineering notebook covering baseline prompting, prompt tuning, and input guardrails.

**Student Experience (3 acts):**
- **Act I — Baseline:** run a starter prompt and measure accuracy.
- **Act II — Power of Prompts:** compare a deliberately bad prompt and your improved prompt.
- **Act III — Making It Robust:** enable `use_guardrails` and observe how invalid inputs are handled.

> This notebook is a **restructure** of the original workshop notebooks. Code cells are reused from the originals with minimal re-ordering to fit the teaching flow.


## Step 1: Install Dependencies


In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt


## Step 2: Load Shared Utilities


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Configure API Client


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# Configuration (same pattern as baseline notebook)

NOTEBOOK_NAME = "02_basic_parsing_and_guardrails"

STUDENT_EMAIL = os.getenv("WORKSHOP_EMAIL", "")

MODEL_NAME = os.getenv("WORKSHOP_BASELINE_MODEL", "gemini-2.5-flash")

STAGE_NAME = "prompt_tuned"

MOCK_MODE = True   # Set False to use real workshop gateway

TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512

USE_GUARDRAILS = False

MAX_WORKERS = 4

## Step 4: Define Reusable Experiment Function


In [ ]:
from esmt_workshop.student_api import process_batch_addresses


def run_experiment(prompt_template: str, *, temperature: float = 0.0, top_p: float = 1.0, top_k: int = 40):
    # Load development labels for prompt iteration.
    dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

    # Use the same model while changing prompt and sampling controls.
    pred_df = process_batch_addresses(
        dev_df,
        email=STUDENT_EMAIL,
        stage='prompt_tuned',
        model=MODEL_NAME,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        max_tokens=MAX_TOKENS,
        custom_prompt_template=prompt_template,
        max_workers=MAX_WORKERS,
    )

    report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)
    return pred_df, report


## Step 5: Run the Starter Prompt


In [ ]:
# Starter prompt editable directly in code.
STARTER_PROMPT_TEMPLATE = '''You are an address parser for AML compliance.

Input address:
{address}

Return strict JSON only using this schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

pred_v1, report_v1 = run_experiment(STARTER_PROMPT_TEMPLATE, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K)

print('v1 summary:', report_v1['summary'])
report_v1['field_metrics']


## Step 6: Run a Deliberately Bad Prompt

This is intentionally low-quality to show how instruction quality impacts results.

In [ ]:
# Deliberately low-quality prompt (for demonstration).
BAD_PROMPT_TEMPLATE = "Parse this address: {address}. Give me JSON."

pred_bad, report_bad = run_experiment(
    BAD_PROMPT_TEMPLATE,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
)

print('bad summary:', report_bad['summary'])
report_bad.get('field_metrics', None)

## Step 6: Run Your Team Prompt


In [ ]:
# Team prompt editable directly in this notebook cell.
STUDENT_PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Extract Town Name, Postal Code, Remaining Address, Country Code (2 characters).
- Country Code must be ISO alpha-2 uppercase.
- Do not invent values.
- Use empty strings if unknown.
'''

pred_student, report_student = run_experiment(
    STUDENT_PROMPT_TEMPLATE,
    temperature=0.0,
    top_p=0.95,
    top_k=50,
)
print('student summary:', report_student['summary'])
report_student['field_metrics']


## Step 7: Analyze and Compare Results

Use these objects to inspect summaries and common errors.

In [ ]:
# Quick comparison (micro_accuracy if present).
def _micro(rep): 
    return (rep.get('summary') or {}).get('micro_accuracy')

print('Starter micro_accuracy:', _micro(report_v1))
print('Bad micro_accuracy:', _micro(report_bad))
print('Student micro_accuracy:', _micro(report_student))

# Optional: inspect mismatches (if available in the report).
if 'mismatches' in report_student:
    display(report_student['mismatches'].head(10))

## Step 7: Save Prompt-Tuning Artifacts


In [ ]:
# Save prompt-tuning artifacts for both prompt versions.
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_v1.to_csv(out_dir / '02_prompt_tuned_v1_predictions.csv', index=False)
save_evaluation_report(report_v1, out_dir / 'report_02_prompt_tuned_v1')

pred_student.to_csv(out_dir / '02_prompt_tuned_student_predictions.csv', index=False)
save_evaluation_report(report_student, out_dir / 'report_02_prompt_tuned_student')

print('Saved prompt-tuning artifacts.')


## Act III — Making It Robust (Guardrails)

The following section is taken from the original **05_input_guardrails** notebook to demonstrate `use_guardrails` behavior.

## Step 3: Student Identity and Generation Controls


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# Notebook metadata used in experiment history logs.
NOTEBOOK_NAME = '05_input_guardrails'

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')

# Exposed generation controls available in every processing call.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8

# Free-text note for this run (optional, appears in history table).
RUN_NOTES = ''

STAGE_NAME = 'two_stage'
MODEL_NAME = 'gemini-2.5-pro'
COUNTRY_MODEL = 'gemini-2.5-flash'
USE_GUARDRAILS = True


## Step 4: Edit Prompt Directly in Notebook


In [ ]:
# Set to False to use built-in stage prompt logic.
USE_CUSTOM_PROMPT = True
PROMPT_LABEL = 'notebook_editable_prompt_v1'
PROMPT_TEMPLATE = '''You are an AML address parser.

Detected country:
{country}

Country format guidance:
{kb_text}

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Use country guidance only for mapping decisions.
- Do not invent missing tokens.
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None
print('Prompt length:', len(PROMPT_TEMPLATE) if custom_prompt else 0)


## Step 5: Run Pipeline


In [ ]:
import time

reference_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/reference_100.csv', dtype=str).fillna('')
input_df = reference_df

# Execute configured stage and measure runtime.
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    input_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=custom_prompt,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    max_workers=MAX_WORKERS,
)
runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))
pred_df.head(5)


## Step 6: Evaluate Output


In [ ]:
report = evaluate_predictions(pred_df, input_df, email=STUDENT_EMAIL)
print(report['summary'])
display(report['field_metrics'])


## Step 7: Save Artifacts and Log the Run


In [ ]:
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / '05_input_guardrails_predictions.csv'
pred_df.to_csv(pred_path, index=False)

report_dir = out_dir / 'report_05_input_guardrails'
if report.get('summary'):
    save_evaluation_report(report, report_dir)

run_record = log_experiment_run(
    output_root=PROJECT_ROOT / 'outputs',
    notebook_name=NOTEBOOK_NAME,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_workers=MAX_WORKERS,
    use_guardrails=USE_GUARDRAILS,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    prompt_template=custom_prompt,
    prompt_label=PROMPT_LABEL if USE_CUSTOM_PROMPT else 'stage_default_prompt',
    runtime_sec=runtime_sec,
    metrics_summary=report.get('summary', {}),
    notes=RUN_NOTES,
    predictions_path=pred_path,
    report_dir=report_dir if report.get('summary') else '',
)

print('Saved predictions:', pred_path)
if report.get('summary'):
    print('Saved report:', report_dir)
print('Logged run_id:', run_record['run_id'])


## Step 8: Prompt/Parameter History Summary


In [ ]:
history_df = load_experiment_history(output_root=PROJECT_ROOT / 'outputs', notebook_name=NOTEBOOK_NAME)
if history_df.empty:
    print('No runs logged yet.')
else:
    raw_cols = [
        'created_at_utc', 'stage', 'model', 'country_model', 'prompt_label',
        'temperature', 'top_p', 'top_k', 'micro_accuracy', 'runtime_sec',
        'row_exact_match', 'prompt_hash', 'notes'
    ]
    display(history_df[raw_cols].head(100))

    summary_df = (
        history_df.groupby(
            ['prompt_label', 'model', 'country_model', 'temperature', 'top_p', 'top_k', 'use_guardrails', 'mock_mode'],
            dropna=False,
        )
        .agg(
            runs=('run_id', 'count'),
            micro_accuracy_mean=('micro_accuracy', 'mean'),
            row_exact_match_mean=('row_exact_match', 'mean'),
            runtime_sec_mean=('runtime_sec', 'mean'),
        )
        .reset_index()
        .sort_values(['micro_accuracy_mean', 'row_exact_match_mean'], ascending=False)
    )

    display(summary_df.head(30))


## Assignment

1. Edit `STUDENT_PROMPT_TEMPLATE` in this notebook and rerun.
2. Compare `micro_accuracy` and mismatch profiles.
3. Move to `03_advanced_model.ipynb`.


## Assignment

1. Tune guardrails versus false rejects.
2. Compare guardrail configurations in the history table.
3. Continue with 06_final_submission_and_validation.ipynb.
